In [1]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, LabelSet, HoverTool
from bokeh.io import output_notebook  # optional if you're in a notebook

# output_notebook()  # uncomment if using a Jupyter notebook

PAGE_WIDTH  = 2720
PAGE_HEIGHT = 936

# Rectangle sizes in pixels
purple_w, purple_h = 112, 63
turq_w, turq_h     = 206, 137

# Filter centers with origin at the CENTER of the figure (x right, y up)
filter_centers = {
    "1":  {"center": (-1248.5, 437.0),  "sensor": "hwk4123", "filter": "BB"},
    "2":  {"center": (-746.5, 437.0),   "sensor": "hwk4123", "filter": "z"},
    "3":  {"center": (-414.5, 437.0),   "sensor": "hwk4123", "filter": "BB"},
    "4":  {"center": (-78.0, 437.0),    "sensor": "hwk4123", "filter": "g"},
    "5":  {"center": (256.0, 437.0),    "sensor": "hwk4123", "filter": "BB"},
    "6":  {"center": (590.5, 437.0),    "sensor": "hwk4123", "filter": "u"},
    "7":  {"center": (924.5, 437.0),    "sensor": "hwk4123", "filter": "BB"},
    "8":  {"center": (1258.5, 437.0),   "sensor": "hwk4123", "filter": "i"},
    "9":  {"center": (-1244.5, 12.0),   "sensor": "imx455" , "filter": "N-II"},
    "10": {"center": (-886.5, 12.0),    "sensor": "imx455" , "filter": "H-alpha"},
    "11": {"center": (-533.5, 12.0),    "sensor": "imx455" , "filter": "H-Beta"},
    "12": {"center": (-176.0, 12.0),    "sensor": "imx455" , "filter": "BB +2w"},
    "13": {"center": (183.5, 12.0),     "sensor": "imx455" , "filter": "i"},
    "14": {"center": (542.5, 12.0),     "sensor": "imx455" , "filter": "g"},
    "15": {"center": (896.5, 12.0),     "sensor": "imx455" , "filter": "r"},
    "16": {"center": (1254.0, 12.0),    "sensor": "imx455" , "filter": "r +1w"},
    "17": {"center": (-1254.0, -399.5), "sensor": "imx455" , "filter": "BB"},
    "18": {"center": (-891.5, -399.5),  "sensor": "imx455" , "filter": "u"},
    "19": {"center": (-532.5, -399.5),  "sensor": "imx455" , "filter": "He II"},
    "20": {"center": (-174.5, -399.5),  "sensor": "imx455" , "filter": "z"},
    "21": {"center": (184.5, -399.5),   "sensor": "imx455" , "filter": "O-III"},
    "22": {"center": (541.5, -399.5),   "sensor": "imx455" , "filter": "BB"},
    "23": {"center": (901.0, -399.5),   "sensor": "imx455" , "filter": "r -1w"},
}

# ---- Build a ColumnDataSource for Bokeh (include filter names) ----
xs, ys, ws, hs, colors, ids, sensors, filter_names = [], [], [], [], [], [], [], []
for fid, props in filter_centers.items():
    cx, cy = props["center"]
    sensor = props["sensor"]
    fname  = props.get("filter", "")

    if sensor == "hwk4123":
        w, h = purple_w, purple_h
        color = "#6f63a6"
    elif sensor == "imx455":
        w, h = turq_w, turq_h
        color = "#12b3b3"
    else:
        raise ValueError(f"Unknown sensor type: {sensor}")

    xs.append(cx)
    ys.append(cy)
    ws.append(w)
    hs.append(h)
    colors.append(color)
    ids.append(fid)
    sensors.append(sensor)
    filter_names.append(fname)

source = ColumnDataSource(data=dict(
    x=xs, y=ys, w=ws, h=hs,
    color=colors,
    fid=ids,
    sensor=sensors,
    filter_name=filter_names,    # <-- added field
))

pad = 50
x_range = (-PAGE_WIDTH/2 - pad,  PAGE_WIDTH/2 + pad)
y_range = (-PAGE_HEIGHT/2 - pad, PAGE_HEIGHT/2 + pad)

# ---- Make the plot ----
p = figure(
    title="WCC Filter Layout",
    x_range=x_range,
    y_range=y_range,
    width=900,
    height=450,
    match_aspect=True,
    tools="pan,wheel_zoom,reset,save",
    toolbar_location="above",
)

# Turn off axes/grid like your matplotlib version
p.axis.visible = False
p.grid.visible = False

# Draw filter rectangles (center-based in Bokeh)
rects = p.rect(
    x="x", y="y", width="w", height="h",
    source=source,
    fill_color="color",
    line_color="black",
    line_width=0.5,
)

# Add ID labels centered in each rectangle
labels = LabelSet(
    x="x", y="y", text="filter_name",
    source=source,
    text_align="center",
    text_baseline="middle",
    text_color="white",
    text_font_size="11pt",
    text_font_style="bold",
)
p.add_layout(labels)

# Draw the big FOV outline (also center-based)
p.rect(
    x=0, y=0, width=PAGE_WIDTH, height=PAGE_HEIGHT,
    fill_alpha=0,
    line_color="black",
    line_width=1.0,
)

# ---- Hover showing filter name (and other fields) ----
hover = HoverTool(renderers=[rects],
                  tooltips=[
                      ("Filter ID", "@fid"),
                      ("Filter name", "@filter_name"),
                      ("Sensor", "@sensor"),
                      ("Center (x,y)", "(@x, @y)"),
                  ])
p.add_tools(hover)

show(p)